<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/9_tools/9_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Built-in Tools Overview

OpenAI provides several built-in tools that you can enable with just a few lines of code. Unlike function calling where you define and implement functions yourself, these tools are hosted by OpenAI - you simply enable them and the model handles the rest.

| Tool | Purpose | Use Case |
|------|---------|----------|
| **Web Search** | Search the internet | Current events, real-time data, fact-checking |
| **Remote MCP** | Connect to external services | Third-party integrations, external APIs |
| **File Search** | Query uploaded documents | Knowledge bases, documentation, RAG |
| **Code Interpreter** | Execute Python code | Data analysis, calculations, visualizations |
| **Computer Use** | Control computer interfaces | GUI automation, web browsing (advanced) |

---

# Setup

## Install Dependencies

In [1]:
# Install the OpenAI library
!pip install -q openai==2.28.0

# Suppress deprecation warnings for cleaner output
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("✅ All dependencies installed!")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✅ All dependencies installed!


## API Key Configuration

You have two methods to provide your API key:

**Method 1 (Recommended)**: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `OPENAI_API_KEY`
4. Value: Your OpenAI API key
5. Enable notebook access

**Method 2 (Fallback)**: Manual input when prompted

Run the cell below to configure authentication:

In [2]:
import os

# Configure OpenAI API key
# Method 1: Try to get API key from Colab secrets (recommended)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    import os
    if os.environ.get('OPENAI_API_KEY'):
        OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
        print("✅ API key loaded from environment variable")
    else:
        # Method 2: Manual input (fallback)
        from getpass import getpass
        print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: OPENAI_API_KEY")
        OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Authentication configured!")

✅ API key loaded from environment variable
✅ Authentication configured!


## Import Libraries

In [3]:
from openai import OpenAI
import json

print("✅ All libraries imported!")

✅ All libraries imported!


## Initialize OpenAI Client

Now let's create a client instance to interact with the OpenAI API:

In [4]:
# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Configure which OpenAI model to use
OPENAI_MODEL = "gpt-5-nano"

print("✅ OpenAI client initialized!")
print(f"Selected Model: {OPENAI_MODEL}")

✅ OpenAI client initialized!
Selected Model: gpt-5-nano


---

# Web Search

The **Web Search** tool allows the LLM to search the internet for up-to-date information. LLMs have a knowledge cutoff date, so they cannot answer questions about recent events, real-time data like weather or stock prices, or newly published information. Web search solves this by letting the model look things up on the internet before answering.

## How It Works

You enable web search by adding `{"type": "web_search_preview"}` to the tools array. When the model needs current information, it automatically searches the web, incorporates the results into its response, and cites the sources so you can verify them. The model decides on its own when a search is needed based on the query.


---

## Example: Basic Web Search


In [5]:
# Enable web search and ask about current events
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search_preview"}],  # Enable web search
    input="What are the latest developments in AI this week?"
)

print("Web Search Results:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

Web Search Results:
Here are the most notable AI developments from this week (around July 1–6, 2026), with exact dates for clarity:

- EU AI Act amendments move forward (Digital Omnibus on AI)
  - The EU Council gave final green light to simplify and streamline AI rules as part of Omnibus VII. Key changes shorten timelines for high-risk AI rules and set new dates: stand-alone high-risk rules apply from Dec 2, 2027, and high-risk rules embedded in products from Aug 2, 2028. The package also tightens content safeguards (prohibiting AI-generated non-consensual sexual content) and adjusts sandbox/transparency timelines. Date: June 29, 2026. Source: EU Council press release. ([consilium.europa.eu](https://www.consilium.europa.eu/en/press/press-releases/2026/06/29/artificial-intelligence-council-gives-final-green-light-to-simplify-and-streamline-rules/pdf/))

- UN launches AI governance effort for “AI for Good” (global coordination)
  - The UN, ITU, and global leaders announced the AI for Go

## Example: Real-Time Information

Web search is especially useful for data that changes constantly, like weather:

In [6]:
# Ask about something that requires real-time data
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search_preview"}],
    input="What is the current weather in Tokyo, Japan?"
)

print("Weather Query Results:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

Weather Query Results:
Current weather in Tokyo, Japan (today, Monday, July 6, 2026): Mostly cloudy, about 80°F (27°C).

Today: Rain and drizzle, high around 80°F (27°C) and low around 66°F (19°C).

Tomorrow (Tuesday, July 7): Mostly cloudy and warmer, high about 82°F (28°C), low around 71°F (21°C).

Would you like hourly details, humidity, or wind information? 


## Example: Fact-Checking with Sources

We can also use web search to verify claims and get sourced information:

In [7]:
# Ask a question that requires verified information
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search_preview"}],
    input="Who won the most recent Nobel Prize in Physics and for what discovery?"
)

print("Nobel Prize Query:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

Nobel Prize Query:
The most recent Nobel Prize in Physics was the 2025 award. It was given on 7 October 2025 to John Clarke, Michel H. Devoret, and John M. Martinis for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit. Clarke is at UC Berkeley; Devoret is affiliated with Yale, UCSB, and Google Quantum AI; Martinis is at UCSB/Qolab. ([nobelprize.org](https://www.nobelprize.org/prizes/physics/2025/press-release/?gsid=a64ea7ab-c21a-4897-8d64-34e1c8f13bba))

Would you like a brief explanation of what “macroscopic quantum tunnelling and energy quantisation in an electric circuit” means?


## Inspecting Web Search Results

The response object contains detailed information about the web search, including the sources used. Let's examine the structure:

In [8]:
# Make a query and inspect the full response
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search_preview"}],
    input="What are the top 3 most popular programming languages in 2025?"
)

print("Response Analysis:")
print("=" * 60)
print(f"\n Main Response:")
print(response.output_text)

# Check for web search results in the output
print(f"\n🔍 Output Items:")
for i, item in enumerate(response.output):
    print(f"  {i+1}. Type: {item.type}")

print("=" * 60)

Response Analysis:

 Main Response:
There isn’t a single universal “top 3” for 2025—the rankings depend on the index you use. Here are the leading triads from two major sources (as of February 2025):

- Tiobe Index (February 2025): Python, C++, Java. ([infoworld.com](https://www.infoworld.com/article/3821294/c-go-and-rust-gaining-popularity-tiobe.html))
- PyPL Index (February 2025): Python, Java, JavaScript. ([infoworld.com](https://www.infoworld.com/article/3821294/c-go-and-rust-gaining-popularity-tiobe.html))

Context from Stack Overflow's 2025 Developer Survey shows JavaScript as the most-used language among respondents (66%), with Python also widely used (57.9%); i.e., JavaScript leads usage in that survey, while Python remains very popular. ([survey.stackoverflow.co](https://survey.stackoverflow.co/2025/?aid=reccHAXyiekn6spGc))

Notes:
- Python leads in both Tiobe and PyPL in early 2025.
- The 2nd and 3rd slots differ by index (C++ and Java in Tiobe; Java and JavaScript in PyPL).


## Example: Combining Web Search with Conversation Context

In [9]:
# Use web search with a multi-turn conversation
messages = [
    {
        "role": "system",
        "content": "You are a helpful research assistant. Use web search to find accurate, current information. Always cite your sources."
    },
    {
        "role": "user",
        "content": "I'm researching electric vehicles. What are the best-selling EVs globally right now?"
    }
]

response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search_preview"}],
    input=messages
)

print("EV Research Results:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

EV Research Results:
Short answer: as of early 2026 the global best-sellers are still heavily dominated by Chinese models led by Tesla’s Model Y. Here are the latest high‑level figures from EV sales trackers (as reported by CleanTechnica, which compiles EV‑Volumes data):

- January 2026 (global, monthly top 3)
  - 1) Tesla Model Y — about 53,000 units
  - 2) Xiaomi YU7 — about 31,000 units
  - 3) Geely Galaxy Xingyuan (EX2 in export markets) — about 31,000 units
  Source: CleanTechnica, Top Selling Electric Vehicles in the World — January 2026. ([cleantechnica.com](https://cleantechnica.com/2026/03/05/top-selling-electric-vehicles-in-the-world-january-2026/))

- February 2026 (global, monthly top 3)
  - 1) Tesla Model Y — about 72,700 units
  - 2) Tesla Model 3 — about 32,200 units
  - 3) Geely Xingyuan (EX2) — about 29,000 units
  Note: Xiaomi YU7 and BYD Song were also in the mix in the top 5/10, but the top 3 for February were Model Y, Model 3, and Xingyuan. ([cleantechnica.com](htt

## When to Use It

| ✅ Good Use Cases | ❌ Not Ideal For |
|-------------------|------------------|
| Current events and news | Historical facts (model already knows) |
| Real-time data (weather, stocks) | Creative writing |
| Recent product releases | General knowledge questions |
| Fact-checking claims | Private/internal information |
| Research with citations | Tasks requiring no external data |

## Pricing Note

Web search incurs additional costs beyond standard token pricing. Each search query has an associated cost, so use it judiciously for queries that truly need current information.

---

# Remote MCP (Model Context Protocol)


The **Model Context Protocol (MCP)** is an open standard introduced by Anthropic in November 2024 and adopted by OpenAI in March 2025. It standardizes how AI systems integrate with external tools, systems, and data sources.

Think of MCP as a **universal adapter** - instead of building custom integrations for every external service, MCP provides a standard interface that any AI model can use to communicate with any MCP-compatible server.

## How Remote MCP Works

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────────┐
│   Your App  │────▶│   OpenAI API    │────▶│  Remote MCP      │
│             │     │   (Responses)   │     │  Server          │
└─────────────┘     └─────────────────┘     └──────────────────┘
                           │                        │
                           │   Tools are listed     │
                           │   and called           │
                           │   automatically        │
                           ▼                        ▼
                    Model decides          External service
                    when to use tools      executes actions
```

1. You specify an MCP server URL in the tools array
2. OpenAI's runtime connects to the server and discovers available tools
3. The model can then call these tools as needed
4. Results are returned and incorporated into the response


The key benefits of MCP are that it requires no custom code (OpenAI handles tool calling automatically), works with any MCP-compatible server through a standardized interface, supports secure authentication via headers, and allows you to connect to multiple servers simultaneously.


## Using the GitMCP Server

For this demo, we'll use the **GitMCP Server** ([gitmcp.io](https://gitmcp.io)) - a free, public MCP server that turns any GitHub project into a searchable MCP server. Specifically, we'll connect to OpenAI's [tiktoken](https://github.com/openai/tiktoken) tokenizer library. This server:
- Provides tools to search and fetch documentation from GitHub repositories
- Requires no API key (works with public repos)



### Example: Fetching Web Content

Let's use the GitMCP server to search the tiktoken repository documentation:

In [10]:
# Using the GitMCP server to search GitHub repository documentation
# This is a free, public MCP server - no API key needed (beyond your OpenAI key)

response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "gitmcp",
            "server_url": "https://gitmcp.io/openai/tiktoken",
            "require_approval": "never"  # Auto-approve tool calls for this demo
        }
    ],
    input="How does tiktoken work? Give me a brief overview."
)

print("MCP GitMCP Results:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

MCP GitMCP Results:
Here’s a concise overview of how tiktoken works:

- What it is: A fast, OpenAI-compatible tokenizer used to convert text to and from token IDs for models like GPT-3.5-turbo and GPT-4. It helps you count tokens, prepare prompts, and estimate costs.

- Model-specific encoding: Each model family uses a specific tokenization scheme (e.g., cl100k_base, p50k_base). tiktoken ships the appropriate encoder for the model you’re targeting.

- How it works at a high level:
  - Normalize the input text to a stable form.
  - Convert the text into a sequence of bytes and apply a byte-level encoding that maps those bytes to tokens.
  - Use a precomputed vocabulary/merge rules (a form of byte-pair encoding) to combine byte-level tokens into higher-level tokens that the model expects.
  - Produce a list of token IDs (encode) or reconstruct text from token IDs (decode).
  - It also provides a token-counting path to quickly estimate how many tokens a piece of text would use.

- Key fea



When using MCP, the response includes information about which tools were discovered and used:

In [11]:
# Make an MCP request and examine the response structure
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "gitmcp",
            "server_url": "https://gitmcp.io/openai/tiktoken",
            "require_approval": "never"
        }
    ],
    input="What encoding models does tiktoken support?"
)

print(" MCP Response Analysis:")
print("=" * 60)

# Examine each item in the output
for i, item in enumerate(response.output):
    print(f"\n Output Item {i+1}:")
    print(f"   Type: {item.type}")

    # Check for MCP-specific items
    if item.type == "mcp_list_tools":
        print(f"   Server: {item.server_label}")
        print(f"   Tools discovered: {len(item.tools)}")
        for tool in item.tools:
            print(f"     - {tool.name}")
    elif item.type == "mcp_call":
        print(f"   Tool called: {item.name}")
    elif item.type == "message":
        print(f"   Content preview: {item.content[0].text[:100]}...")

print("\n" + "=" * 60)
print("\n Final Response:")
print(response.output_text)

 MCP Response Analysis:

 Output Item 1:
   Type: mcp_list_tools
   Server: gitmcp
   Tools discovered: 4
     - fetch_tiktoken_documentation
     - search_tiktoken_documentation
     - search_tiktoken_code
     - fetch_generic_url_content

 Output Item 2:
   Type: reasoning

 Output Item 3:
   Type: mcp_call
   Tool called: fetch_tiktoken_documentation

 Output Item 4:
   Type: reasoning

 Output Item 5:
   Type: mcp_call
   Tool called: search_tiktoken_documentation

 Output Item 6:
   Type: reasoning

 Output Item 7:
   Type: mcp_call
   Tool called: search_tiktoken_code

 Output Item 8:
   Type: reasoning

 Output Item 9:
   Type: mcp_call
   Tool called: fetch_generic_url_content

 Output Item 10:
   Type: reasoning

 Output Item 11:
   Type: message
   Content preview: tiktoken ships the following built-in encoding models:

- gpt2
- r50k_base
- p50k_base
- p50k_edit
-...


 Final Response:
tiktoken ships the following built-in encoding models:

- gpt2
- r50k_base
- p50k_base
- p5

## Example: Searching for Code Patterns

We can also search for specific code patterns and usage examples in the repository:

In [12]:
# Use MCP to search code in a GitHub repository
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[
        {
            "type": "mcp",
            "server_label": "gitmcp",
            "server_url": "https://gitmcp.io/openai/tiktoken",
            "require_approval": "never"
        }
    ],
    input="Search the tiktoken codebase and explain how to use tiktoken to count tokens for GPT-4. Include a code example."
)

print(" Code Search Results:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

 Code Search Results:
Here's how to count tokens for GPT-4 using the tiktoken library, based on the tiktoken codebase/docs and the OpenAI Cookbook examples.

What you need to know
- For GPT-4 (and GPT-4o, etc.), use a model-specific encoder via encoding_for_model. This selects the tokenization that matches the model's usage (usually based on the cl100k_base encoding for GPT-4 families).
  - Example: enc = tiktoken.encoding_for_model("gpt-4")
  - Example: enc = tiktoken.encoding_for_model("gpt-4o")
- For simple prompts (a single string), count tokens with len(enc.encode(text)).
- For chat-style messages (list of messages with roles), token counting has per-message overhead. OpenAI’s recommended approach adds a small fixed number of tokens per message, plus a final priming token. The OpenAI Cookbook shows a common helper to count tokens for messages.

What the code looks like

1) Count tokens for a plain string (prompt)

Python
import tiktoken

# Get the encoder for GPT-4 (you can also u

## MCP in Production

While we used GitMCP for this demo, in production you can connect to many other MCP servers:

| MCP Server | Purpose | Requires API Key |
|------------|---------|------------------|
| GitMCP | GitHub repository access | No (public repos) |
| DeepWiki| Wiki/knowledge search | No |
| Stripe | Payment processing | Yes (Stripe) |
| Shopify| E-commerce data | Yes (Shopify) |
| Zapier | Workflow automation | Yes (Zapier) |
| Twilio | Messaging/SMS | Yes (Twilio) |



For servers that require authentication, you can pass headers:

```python
{
    "type": "mcp",
    "server_label": "my_service",
    "server_url": "https://mcp.myservice.com/sse",
    "headers": {
        "Authorization": "Bearer YOUR_API_KEY"
    },
    "require_approval": "never"
}
```

## Pricing Note

There is **no additional cost** to call remote MCP servers through OpenAI - you're simply billed for the output tokens as usual.

---

# File Search (Vector Store RAG)



**File Search** is OpenAI's built-in Retrieval-Augmented Generation (RAG) solution. It allows you to:

1. Upload your own documents (PDFs, text files, etc.)
2. OpenAI automatically chunks, embeds, and indexes them
3. Query your documents using natural language
4. Get answers grounded in your specific content

This is essentially **hosted RAG** - OpenAI handles all the complexity of:
- Text chunking
- Embedding generation
- Vector storage
- Similarity search

## How It Works

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────────┐
│  Your Files │────▶│  Vector Store   │────▶│  File Search     │
│  (upload)   │     │  (auto-indexed) │     │  Tool            │
└─────────────┘     └─────────────────┘     └──────────────────┘
                           │                        │
                           │   Chunks stored        │  User query
                           │   as embeddings        │  searches
                           ▼                        ▼
                    Automatic               Relevant chunks
                    processing              added to context
```

This means you do not need to set up your own vector database. Files are chunked and embedded automatically, the API is simple (just upload and query), and it scales to handle large document collections.


## Step 1: Load Demo Files

Let's load three text files about a fictional company that will serve as our knowledge base.

In [13]:
demo_files = ["company_policies.txt", "product_catalog.txt", "faq.txt"]

for filename in demo_files:
    with open(filename, "r") as f:
        content = f.read()

    # These prints are now correctly inside the for loop
    print(f"{filename} ({len(content)} characters)")
    print("-" * 60)
    print(content[:200] + "..." if len(content) > 200 else content)
    print("=" * 60 + "\n")

print(f"✅ All {len(demo_files)} files loaded and ready for upload to OpenAI!")

company_policies.txt (4878 characters)
------------------------------------------------------------
TECHSTART INC. - COMPANY POLICIES AND GUIDELINES

Document Version: 2.1
Last Updated: January 2025

TABLE OF CONTENTS
-----------------
1. Remote Work...

product_catalog.txt (8724 characters)
------------------------------------------------------------
TECHSTART INC. - PRODUCT CATALOG 2025

Welcome to the TechStart Product Catalog. We offer innovative software solutions
for businesses of all sizes. This catalog...

faq.txt (11445 characters)
------------------------------------------------------------
TECHSTART INC. - FREQUENTLY ASKED QUESTIONS (FAQ)

Last Updated: January 2025

This document contains answers to the most commonly asked questions ab...

✅ All 3 files loaded and ready for upload to OpenAI!


## Step 2: Upload Files to OpenAI

Now we upload the files to OpenAI's storage. This makes them available for indexing.

In [14]:
# Upload files to OpenAI
file_paths = ["company_policies.txt", "product_catalog.txt", "faq.txt"]
uploaded_files = []

print(" Uploading files to OpenAI...")
print("=" * 60)

for file_path in file_paths:
    try:
        with open(file_path, "rb") as f:
            file = client.files.create(
                file=f,
                purpose="assistants"  # Purpose for file search
            )
            uploaded_files.append(file)
            print(f"✅ Uploaded: {file_path}")
            print(f"   File ID: {file.id}")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        print("   Please upload the demo files first (run the previous cell)")

print("=" * 60)
print(f"\n Total files uploaded: {len(uploaded_files)}")

 Uploading files to OpenAI...


✅ Uploaded: company_policies.txt
   File ID: file-GH8aMGfq4fjwdAUfvT7vKN


✅ Uploaded: product_catalog.txt
   File ID: file-8BH7adf3yhsHYAt3TXA47k


✅ Uploaded: faq.txt
   File ID: file-BBy5dMqJcFoyna3F2wrmqE

 Total files uploaded: 3


## Step 3: Create a Vector Store

A Vector Store is a container that holds your indexed files. When you add files to a vector store, OpenAI automatically:
1. Chunks the text into smaller pieces
2. Creates embeddings for each chunk
3. Stores everything for efficient retrieval

In [15]:
# Create a vector store
print(" Creating Vector Store...")

vector_store = client.vector_stores.create(
    name="TechStart Knowledge Base"
)

print(f"✅ Vector Store created!")
print(f"   ID: {vector_store.id}")
print(f"   Name: {vector_store.name}")
print(f"   Status: {vector_store.status}")

 Creating Vector Store...


✅ Vector Store created!
   ID: vs_6a4b5b50fa9c8191864bea644f4b8ac5
   Name: TechStart Knowledge Base
   Status: completed


## Step 4: Add Files to the Vector Store

Now we attach our uploaded files to the vector store for indexing.

In [16]:
import time

# Add files to the vector store
print(" Adding files to Vector Store...")
print("=" * 60)

for file in uploaded_files:
    client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file.id
    )
    print(f"   Added: {file.filename}")

# Wait for processing to complete
print("\n⏳ Waiting for files to be processed...")

while True:
    vs_status = client.vector_stores.retrieve(vector_store.id)
    file_counts = vs_status.file_counts

    print(f"   Status: {file_counts.completed}/{file_counts.total} files processed", end="\r")

    if file_counts.completed == file_counts.total:
        break

    time.sleep(1)

print("\n" + "=" * 60)
print(f"\n✅ All files processed and indexed!")
print(f"   File counts: {file_counts.completed} completed, {file_counts.total} total")

 Adding files to Vector Store...


   Added: company_policies.txt


   Added: product_catalog.txt


   Added: faq.txt

⏳ Waiting for files to be processed...


   Status: 3/3 files processed

✅ All files processed and indexed!
   File counts: 3 completed, 3 total


## Step 5: Query Your Documents

Now comes the exciting part - querying your documents using natural language!

In [17]:
# Store the vector store ID for use in queries
VECTOR_STORE_ID = vector_store.id

def search_documents(query):
    """Search documents using the file search tool."""
    response = client.responses.create(
        model=OPENAI_MODEL,
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [VECTOR_STORE_ID]
            }
        ],
        input=query
    )
    return response.output_text

print("✅ Search function ready!")
print(f" Using Vector Store: {VECTOR_STORE_ID}")

✅ Search function ready!
 Using Vector Store: vs_6a4b5b50fa9c8191864bea644f4b8ac5


## Example Queries

Let's test our document search with various questions:

In [18]:
# Query 1: HR Policy Question
print("Query 1: Vacation Policy")
print("=" * 60)
result = search_documents("How many vacation days do employees get per year?")
print(result)
print("=" * 60)

Query 1: Vacation Policy


At TechStart Inc., vacation (annual leave) is tenure-based:
- 0–2 years of service: 15 days per year
- 3–5 years of service: 20 days per year
- 6+ years of service: 25 days per year

Source: TechStart Inc. company policies (Vacation and Time Off) 


In [19]:
# Query 2: Product Information
print("Query 2: Product Pricing")
print("=" * 60)
result = search_documents("What is the pricing for DataFlow Pro?")
print(result)
print("=" * 60)

Query 2: Product Pricing


DataFlow Pro pricing (USD):
- Starter: $499/month (up to 5 users, 100GB data)
- Professional: $1,499/month (up to 25 users, 1TB data)
- Enterprise: $4,999/month (unlimited users, 10TB data)
- Custom: Contact sales for larger deployments

Notes:
- All prices are listed in USD. 
- All TechStart products offer a 14-day free trial. 
- Annual billing is available and includes a 20% discount off the monthly price. 

Would you like me to estimate annual vs. monthly costs for a specific team size?


In [20]:
# Query 3: Technical Question
print("Query 3: Security Features")
print("=" * 60)
result = search_documents("What MFA methods are supported by SecureAuth 360?")
print(result)
print("=" * 60)

Query 3: Security Features


SecureAuth 360 supports several MFA methods:

- Authenticator apps (Google Authenticator, Microsoft Authenticator, Authy)
- Hardware security keys (YubiKey, Titan)
- SMS verification (note: not recommended for high-security environments)
- Email verification
- Push notifications via the SecureAuth mobile app
- Biometric authentication (on devices that support it)

Source: SecureAuth 360 FAQ 


In [21]:
# Query 4: FAQ Question
print("Query 4: Account Management")
print("=" * 60)
result = search_documents("How do I reset my password if I forgot it?")
print(result)
print("=" * 60)

Query 4: Account Management


Here’s the standard password-reset flow you’d use for TechStart accounts:

- Go to the login page.
- Click “Forgot Password.”
- Enter your email address.
- Check your email for the password reset link (the link expires after 24 hours).
- Click the link and set a new password.

If you don’t receive the reset email, check your spam folder. If it still doesn’t arrive, contact support for help.  

Tips for after you reset (helpful to follow): 
- Use a strong password (minimum 12 characters, with upper and lower case letters, numbers, and symbols). 
- Don’t reuse a recent password (policy says no reuse for the last 10). 
- Consider enabling two-factor authentication (2FA) for added security, since 2FA is required for all company systems. 

If you still have trouble, you can reach TechStart support at support@techstart.com. 


In [22]:
# Query 5: Cross-document query
print("Query 5: Cross-Document Search")
print("=" * 60)
result = search_documents("What are the remote work requirements and what equipment is provided to remote workers?")
print(result)
print("=" * 60)

Query 5: Cross-Document Search


Here’s what TechStart’s remote work setup covers, based on the policy documents:

Remote work requirements
- Eligible up to 3 days per week of remote work, with manager approval. Some roles may be fully remote and require explicit department-head approval. 
- Requirements for remote work:
  - Stable internet connection (minimum 50 Mbps recommended)
  - Dedicated workspace free from distractions
  - Availability during core hours (10:00 AM – 3:00 PM local time)
  - Participation in all scheduled video meetings with the camera on
- Requests must be submitted via the HR portal at least 48 hours in advance; managers have discretion to approve/deny based on team needs. 

Equipment provided to remote workers
- Standard Equipment Package:
  - Laptop (MacBook Pro or Dell XPS, depending on role)
  - External monitor (24" minimum)
  - Keyboard and mouse
  - Headset for video calls
  - Software licenses provided (Microsoft 365, Slack, Zoom, GitHub, Figma, Salesforce) 
- Equipment refresh cycle:
 

## Cleanup: Delete Vector Store (Optional)

When you're done, you can delete the vector store to avoid storage charges. Remember, the first 1GB is free, but it's good practice to clean up resources you're not using.

In [23]:
# Uncomment and run this cell to delete the vector store
# This will delete all indexed data - only do this when you're done!

#client.vector_stores.delete(VECTOR_STORE_ID)
#print(f" Vector Store {VECTOR_STORE_ID} deleted!")

print("💡 To delete the vector store, uncomment the lines above and run this cell.")
print(f"   Vector Store ID: {VECTOR_STORE_ID}")

💡 To delete the vector store, uncomment the lines above and run this cell.
   Vector Store ID: vs_6a4b5b50fa9c8191864bea644f4b8ac5


## File Search Pricing

| Component | Cost |
|-----------|------|
| Vector Storage | First 1GB free, then \$0.10  / GB /day |
| Search Queries | \$2.50 per 1,000 queries |

For a tutorial with small files like ours (< 30KB total), storage is essentially free. You only pay for the queries. For current pricing, check the official website.

---

# Code Interpreter

**Code Interpreter** allows the LLM to write and execute Python code in a secure, sandboxed environment. This is useful for data analysis (processing CSV files, calculating statistics), mathematical calculations, creating visualizations, file processing, and solving programming challenges.

## How It Works

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────────┐
│   User      │────▶│   LLM writes    │────▶│  Sandboxed       │
│   Query     │     │   Python code   │     │  Python Runtime  │
└─────────────┘     └─────────────────┘     └──────────────────┘
                           │                        │
                           │   Code generation      │  Code execution
                           │                        │
                           ▼                        ▼
                    Model refines            Results returned
                    if errors occur          (data, charts, files)
```

The code runs in an isolated sandbox with access to common libraries like pandas, numpy, and matplotlib. If the code fails, the model can debug and retry automatically. It can also generate output files (images, CSVs) that you can download.

---

## Example: Mathematical Calculations

When using Code Interpreter, you must specify a `container` parameter that tells OpenAI how to provision the sandboxed Python environment where your code will run. The simplest option is `{"type": "auto"}`, which lets OpenAI automatically create and manage the container for you. You can also pass a specific container ID if you need a pre-configured environment with custom packages or uploaded files.

In [24]:
# Use code interpreter for complex calculations
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="Calculate the first 20 Fibonacci numbers and find their sum. Also calculate the golden ratio approximation using the last two numbers."
)

print(" Mathematical Calculation:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

 Mathematical Calculation:
Using the common convention F1 = 1, F2 = 1:

- First 20 Fibonacci numbers:
  1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765

- Sum of these 20 numbers: 17710

- Golden ratio approximation using the last two numbers (F20/F19):
  6765 / 4181 ≈ 1.6180339631667064

Note: The true golden ratio φ ≈ 1.618033988749895, so this is a very close approximation.

If you instead start with 0, 1 (i.e., include 0 as the first term), the first 20 numbers would end at 4181, with sum 10945, and the last-two ratio would be 4181/2584 ≈ 1.618034055727554.




Let's examine what happens behind the scenes when Code Interpreter runs:

In [25]:
# Make a request and inspect the code that was executed
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="Generate 100 random numbers between 1 and 1000, then calculate the mean, median, standard deviation, and find the top 5 largest numbers."
)

print(" Code Interpreter Analysis:")
print("=" * 60)

# Examine the output items to see the code execution
for i, item in enumerate(response.output):
    print(f"\n Output Item {i+1}: {item.type}")

    if item.type == "code_interpreter_call":
        print(f"\n    Code Executed:")
        print("   " + "-" * 40)
        # The code is in the input field
        if hasattr(item, 'input'):
            for line in item.input.split('\n'):
                print(f"   {line}")
        print("   " + "-" * 40)

        # Show the output/result
        if hasattr(item, 'output'):
            print(f"\n    Output:")
            print(f"   {item.output[:500]}..." if len(str(item.output)) > 500 else f"   {item.output}")

print("\n" + "=" * 60)
print("\n Final Response:")
print(response.output_text)

 Code Interpreter Analysis:

 Output Item 1: reasoning

 Output Item 2: code_interpreter_call

    Code Executed:
   ----------------------------------------
   ----------------------------------------

 Output Item 3: reasoning

 Output Item 4: message


 Final Response:
Here are 100 random integers between 1 and 1000, followed by the requested calculations.

Generated numbers:
389, 379, 374, 710, 567, 806, 515, 137, 584, 40
908, 221, 745, 612, 981, 463, 152, 274, 888, 999
65, 210, 432, 777, 66, 543, 111, 903, 492, 256
68, 821, 999, 173, 846, 519, 462, 238, 301, 950
12, 678, 345, 501, 742, 928, 333, 254, 614, 703
91, 402, 168, 731, 879, 256, 600, 712, 15, 984
130, 55, 764, 203, 466, 789, 517, 64, 999, 477
29, 812, 999, 214, 368, 701, 944, 552, 390, 26
501, 77, 813, 456, 508, 645, 372, 919, 601, 234
905, 737, 481, 342, 993, 155, 678, 892, 248, 700

Calculated statistics:
- Mean: 464.06
- Median: 423.0
- Standard deviation (population, ddof=0): 275.431
- Standard deviation (sample, ddof

## Example: Data Analysis with Sample Data

Code Interpreter is especially powerful for data analysis tasks:

In [26]:
# Create sample sales data for analysis
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="""Create a sample dataset of monthly sales data for a company over 2 years (24 months).
    Include columns: month, sales_amount, expenses, and region (North, South, East, West).
    Then analyze the data to find:
    1. Total sales and expenses per region
    2. Average monthly profit (sales - expenses)
    3. The best and worst performing months
    4. Which region has the highest profit margin"""
)

print("Sales Data Analysis:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

Sales Data Analysis:
Here is a 2-year sample dataset (24 months) with columns: month, region, sales_amount, expenses. Currency is USD.

Dataset
month,region,sales_amount,expenses
2024-01,North,12000,8000
2024-02,North,15000,9000
2024-03,North,13000,8500
2024-04,North,14000,8200
2024-05,South,17000,9800
2024-06,South,16000,9000
2024-07,South,15500,8600
2024-08,South,16500,9400
2024-09,East,18000,11000
2024-10,East,17500,10500
2024-11,East,19000,11200
2024-12,East,20000,11500
2025-01,West,21000,12000
2025-02,West,20500,11850
2025-03,West,21500,12500
2025-04,West,22000,12800
2025-05,North,12500,8500
2025-06,North,13500,8700
2025-07,North,14500,8600
2025-08,North,14000,8600
2025-09,South,15000,9700
2025-10,South,15500,9800
2025-11 East,East,16500,10000
2025-12 East,East,17000,10200

Analysis

1) Total sales and expenses per region
- East: Sales 108,000; Expenses 64,400; Profit 43,600
- North: Sales 108,500; Expenses 68,100; Profit 40,400
- South: Sales 95,500; Expenses 56,300; Profit 39,20

## Example: Solving Programming Problems

Code Interpreter can also solve programming challenges by writing and testing code:

In [27]:
# Ask code interpreter to solve a programming challenge
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="""Write a Python function to check if a number is a prime number.
    Then use it to find all prime numbers between 1 and 100.
    Finally, calculate what percentage of numbers from 1-100 are prime."""
)

print(" Prime Number Analysis:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

 Prime Number Analysis:
Here’s a simple Python solution.

Code:

def is_prime(n):
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

def primes_up_to(limit):
    return [n for n in range(2, limit + 1) if is_prime(n)]

# Find primes between 1 and 100
primes_1_100 = primes_up_to(100)
percentage = len(primes_1_100) / 100 * 100

print("Primes between 1 and 100:", primes_1_100)
print("Number of primes:", len(primes_1_100))
print("Percentage of numbers from 1 to 100 that are prime: {:.2f}%".format(percentage))

Expected results when running the code:

Primes between 1 and 100: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]
Number of primes: 25
Percentage of numbers from 1 to 100 that are prime: 25.00%


## Example: Working with Uploaded Files

Code Interpreter can also process files you upload. Let's create a sample CSV and analyze it:

In [28]:
# Load and preview the employee data
import csv

with open("employee_data.csv", "r") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"✅ employee_data.csv loaded with {len(rows)} records")
print("\nPreview:")
for row in rows[:5]:
    print(f"  {dict(row)}")

✅ employee_data.csv loaded with 50 records

Preview:
  {'employee_id': 'EMP1000', 'department': 'Engineering', 'salary': '95000', 'years_experience': '8', 'performance_score': '4.2'}
  {'employee_id': 'EMP1001', 'department': 'Sales', 'salary': '72000', 'years_experience': '3', 'performance_score': '3.8'}
  {'employee_id': 'EMP1002', 'department': 'Marketing', 'salary': '68000', 'years_experience': '5', 'performance_score': '4.0'}
  {'employee_id': 'EMP1003', 'department': 'HR', 'salary': '61000', 'years_experience': '2', 'performance_score': '3.5'}
  {'employee_id': 'EMP1004', 'department': 'Finance', 'salary': '88000', 'years_experience': '12', 'performance_score': '4.5'}


Let's upload the CSV file to OpenAI so that Code Interpreter can access it:

In [29]:
# Upload the CSV to OpenAI
with open('employee_data.csv', 'rb') as f:
    csv_file = client.files.create(
        file=f,
        purpose='assistants'
    )

print(f"✅ Uploaded CSV file")
print(f"   File ID: {csv_file.id}")

✅ Uploaded CSV file
   File ID: file-DsV92ELojx8W56fbxDTk6g


We can ask Code Interpreter to analyze the data. The model will read the CSV, write Python code to process it, and return the results:

In [30]:
# Use code interpreter to analyze the uploaded file
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto",
                "file_ids": [csv_file.id]
            }
        }
    ],
    input="""Analyze the employee data CSV file. Please provide:
1. Basic statistics (count, mean, median) for salary and years_experience
2. Average salary by department
3. Correlation between years_experience and salary
4. Top 5 highest paid employees
5. Which department has the highest average performance score?"""
)

print(" Employee Data Analysis:")
print("=" * 60)
print(response.output_text)
print("=" * 60)

 Employee Data Analysis:
Here are the analyzed results for the 50-employee CSV data:

1) Basic statistics
- Salary
  - Count: 50
  - Mean: $83,340
  - Median: $77,500
- Years of experience
  - Count: 50
  - Mean: 6.8 years
  - Median: 6.0 years

2) Average salary by department
- Engineering: $110,500
- Finance: $94,700
- Sales: $73,800
- Marketing: $71,700
- HR: $66,000

3) Correlation between years_experience and salary
- Pearson correlation: ~0.95 (strong positive relationship)

4) Top 5 highest paid employees
- EMP1020 | Engineering | $130,000 | 18 years | 4.9
- EMP1035 | Engineering | $125,000 | 16 years | 4.8
- EMP1005 | Engineering | $120,000 | 15 years | 4.8
- EMP1025 | Engineering | $115,000 | 13 years | 4.7
- EMP1010 | Engineering | $110,000 | 11 years | 4.6

5) Department with the highest average performance score
- Engineering: average performance score ≈ 4.59

If you’d like, I can export these results to a CSV or generate a quick visualization (e.g., salary by department ba

## Code Interpreter Pricing

| Component | Cost |
|-----------|------|
| Container Session | $0.03 per session |
| Session Duration | Active for 1 hour, 30-minute idle timeout |

Code Interpreter charges a small fee per container session. A session starts when you make your first request and stays active for up to 1 hour, with a 30-minute idle timeout. If you make multiple requests within the same session, you are not charged again. After the session ends, there are no further charges. For current pricing, check the official website.

---

# Computer Use

**Computer Use** (also called Computer-Using Agent or CUA) is OpenAI's tool that allows AI models to interact with graphical user interfaces - just like a human would. The model can view screenshots of a computer screen, control mouse movements and clicks, type on a virtual keyboard, and navigate websites, fill forms, and interact with applications.

## How It Works

```
┌─────────────┐     ┌─────────────────┐     ┌──────────────────┐
│  Screenshot │────▶│   CUA Model     │────▶│  Action Output   │
│  (pixels)   │     │   (analyzes)    │     │  (click, type)   │
└─────────────┘     └─────────────────┘     └──────────────────┘
       ▲                                           │
       │                                           │
       └───────────────────────────────────────────┘
                    Loop until task complete
```

1. Your application captures a screenshot
2. The screenshot is sent to the CUA model
3. The model analyzes the image and decides what action to take
4. It returns an action (click coordinates, text to type, etc.)
5. Your application executes the action and captures a new screenshot
6. The loop continues until the task is complete

Computer Use is vision-based (works with any GUI, no special APIs needed), adaptive (handles errors and unexpected situations), and supports multi-step workflows across websites, desktop apps, and more.

---

## Implementation

Computer Use is generally available and works with **GPT-5.4** and newer models. There are three ways to integrate it: using OpenAI's built-in computer use loop via the Responses API, layering custom tools on existing automation setups, or mixing visual and programmatic interaction in code-execution environments.

Here is what a Computer Use API call looks like (not executable in this notebook - requires a browser or VM environment):

```python
# NOTE: This is for reference only - requires special infrastructure

response = client.responses.create(
    model="gpt-5.4",
    tools=[
        {
            "type": "computer_use_preview",
            "display_width": 1920,
            "display_height": 1080,
            "environment": "browser"  # or "desktop"
        }
    ],
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Go to google.com and search for 'OpenAI API documentation'"
                },
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": "<base64-encoded-screenshot>"
                    }
                }
            ]
        }
    ]
)
```

**Important:** Always run Computer Use in an isolated browser or VM, keep a human in the loop for high-impact actions, and treat page content as untrusted input.

## Use Cases

Computer Use is ideal for browser automation (form filling, web scraping, testing), interacting with legacy desktop applications, automating repetitive data entry tasks, end-to-end UI testing without writing test scripts, and building personal assistants that can book travel, manage calendars, or shop online.

## Resources

- [OpenAI Computer-Using Agent](https://openai.com/index/computer-using-agent/)
- [OpenAI CUA Sample App](https://github.com/openai/openai-cua-sample-app)
- [Computer Use Documentation](https://developers.openai.com/api/docs/guides/tools-computer-use)


---

# Summary and Best Practices

## Tool Comparison Matrix

| Tool | Best For | Complexity | Cost |
|------|----------|------------|------|
| **Web Search** | Current events, real-time data | Low | Per query |
| **Remote MCP** | External integrations | Medium | Tokens only |
| **File Search** | Document Q&A, knowledge bases | Medium | Storage + queries |
| **Code Interpreter** | Data analysis, calculations | Low | Per session |
| **Computer Use** | GUI automation, complex workflows | High | Premium pricing |

## Best Practices

### 1. Choose the Right Tool

```
Need current information?      → Web Search
Need external service data?    → Remote MCP
Need to query your documents?  → File Search
Need calculations/analysis?    → Code Interpreter
Need GUI interaction?          → Computer Use
Need custom logic?             → Function Calling (Notebook 11)
```

### 2. Combine Tools When Needed

Sometimes a single tool is not enough. For example, you might need to search the web for current data and then use Code Interpreter to analyze it. You can enable multiple tools in a single request:

You can enable multiple tools in a single request:

```python
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[
        {"type": "web_search_preview"},
        {"type": "code_interpreter"}
    ],
    input="Search for the latest stock price of Apple and calculate the P/E ratio."
)
```

### 3. Consider Costs

Different tools have different cost structures. Web Search and Code Interpreter charge per use, File Search has storage costs (first 1GB free), Remote MCP is included in token costs, and Computer Use is the most expensive option. Always consider whether the task justifies the cost, especially at scale. For current pricing details, check the [OpenAI pricing page](https://openai.com/pricing).



---

## Additional Resources

- [OpenAI Tools Documentation](https://platform.openai.com/docs/guides/tools)
- [OpenAI Cookbook](https://cookbook.openai.com/)
- [MCP Tool Guide](https://cookbook.openai.com/examples/mcp/mcp_tool_guide)
- [File Search Guide](https://platform.openai.com/docs/guides/tools-file-search)
- [Code Interpreter Guide](https://platform.openai.com/docs/guides/tools-code-interpreter)

